# M4/M5: CIFAR-100 ベースライン CNN を作る

## このノートブックでできること
- PyTorch で CIFAR-100 データセットをロードします
- シンプルな畳み込みニューラルネットワーク（CNN）をゼロから構築します
- 交差エントロピー損失（Cross-Entropy Loss）と SGD で学習します
- 学習曲線（Learning Curve）を可視化して訓練状況を確認します

## 所要時間の目安
約 60〜90 分（GPU ランタイムで学習込み）

## 対応するサイトのモジュール
**M4: 学習のしくみ** / **M5: 過学習との戦い**（授業課題2のベースラインに直結）

## 実行環境
- **GPU ランタイム推奨**。Colab メニュー「ランタイム → ランタイムのタイプを変更 → T4 GPU」を選択してください。
- CPU でも動きますが、1エポックあたり数分かかります。

## 注意
**事前学習済みモデル（pretrained model）は使用禁止**（授業課題2の条件）。すべての重みをランダム初期化から学習します。


## 1. ライブラリのインポートと環境確認

必要なライブラリをインポートし、GPU が使えるか確認します。


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# 乱数シードの固定（再現性のため）
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# デバイスの確認
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用デバイス: {device}")
print(f"PyTorch バージョン: {torch.__version__}")
print(f"CUDA 使用可能: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2. CIFAR-100 データセットのロード

CIFAR-100 は 100 クラス・60,000 枚の 32×32 カラー画像データセットです。訓練データ（Train set）50,000 枚と テストデータ（Test set）10,000 枚に分かれています。

前処理（Preprocessing）として:
- テンソルへの変換（ToTensor）: ピクセル値 [0, 255] → [0.0, 1.0]
- 正規化（Normalize）: チャンネルごとに平均・標準偏差で正規化


In [ ]:
# CIFAR-100 の平均・標準偏差（チャンネル別）
CIFAR100_MEAN = (0.5071, 0.4865, 0.4409)
CIFAR100_STD  = (0.2673, 0.2564, 0.2762)

# 前処理の定義
transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

# データセットのダウンロードとロード（初回のみダウンロードが走ります）
trainset = torchvision.datasets.CIFAR100(
    root='./data', train=True, download=True, transform=transform_train
)
testset = torchvision.datasets.CIFAR100(
    root='./data', train=False, download=True, transform=transform_test
)

# データローダー（ミニバッチ学習用）
BATCH_SIZE = 128

trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2
)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

print(f"訓練データ数: {len(trainset):,} 枚")
print(f"テストデータ数: {len(testset):,} 枚")
print(f"クラス数: {len(trainset.classes)}")
print(f"ミニバッチサイズ (Batch Size): {BATCH_SIZE}")
print(f"1エポックあたりのイテレーション数: {len(trainloader)}")


## 3. サンプル画像の表示

データが正しくロードされているか、いくつか表示して確認します。


In [ ]:
# 非正規化関数（表示用）
def denormalize(tensor, mean=CIFAR100_MEAN, std=CIFAR100_STD):
    t = tensor.clone()
    for i, (m, s) in enumerate(zip(mean, std)):
        t[i] = t[i] * s + m
    return t.clamp(0, 1)

# サンプル表示
dataiter = iter(trainloader)
images, labels = next(dataiter)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    img = denormalize(images[i])
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(trainset.classes[labels[i]], fontsize=7)
    ax.axis('off')
plt.suptitle("CIFAR-100 サンプル画像", fontsize=12)
plt.tight_layout()
plt.show()
print(f"画像サイズ: {images.shape}  (バッチ数, チャンネル数, 高さ, 幅)")


## 4. ベースライン CNN の定義

以下の構成のシンプルな CNN を定義します。

```
入力: 3 × 32 × 32
↓ Conv(3→32, 3×3) → BatchNorm → ReLU → MaxPool(2×2)
↓ 32 × 16 × 16
↓ Conv(32→64, 3×3) → BatchNorm → ReLU → MaxPool(2×2)
↓ 64 × 8 × 8  （ただし padding=1 なし の場合は 64 × 6 × 6 → 3 × 3）
↓ Conv(64→128, 3×3, padding=1) → BatchNorm → ReLU → MaxPool(2×2)
↓ 128 × 4 × 4
↓ Flatten
↓ Linear(2048 → 512) → ReLU
↓ Linear(512 → 100)
出力: 100 クラスのロジット (Logit)
```

畳み込み（Convolution）は「画像の局所パターンを検出するフィルタ」です。MaxPool（最大プーリング）は「近傍の最大値を取って空間サイズを半分に圧縮」します。


In [ ]:
class BaselineCNN(nn.Module):
    """CIFAR-100 ベースライン CNN。
    事前学習なし、ゼロから学習するシンプルなモデル。
    """

    def __init__(self, num_classes=100):
        super().__init__()

        # 特徴抽出部（Feature Extractor）
        self.features = nn.Sequential(
            # ブロック 1: 3 × 32 × 32 → 32 × 16 × 16
            nn.Conv2d(3, 32, kernel_size=3, padding=1),   # 畳み込み層
            nn.BatchNorm2d(32),                            # バッチ正規化
            nn.ReLU(inplace=True),                         # 活性化関数 ReLU
            nn.MaxPool2d(kernel_size=2, stride=2),         # 最大プーリング

            # ブロック 2: 32 × 16 × 16 → 64 × 8 × 8
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # ブロック 3: 64 × 8 × 8 → 128 × 4 × 4
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # 分類部（Classifier）: 128 × 4 × 4 → 100
        self.classifier = nn.Sequential(
            nn.Flatten(),                          # 平坦化
            nn.Linear(128 * 4 * 4, 512),          # 全結合層
            nn.ReLU(inplace=True),
            nn.Linear(512, num_classes),           # 出力層（100クラス）
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# モデルのインスタンス化とデバイスへの転送
model = BaselineCNN(num_classes=100).to(device)

# パラメータ数の確認
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"総パラメータ数: {total_params:,}")
print(f"学習可能パラメータ数: {trainable_params:,}")
print()
print(model)


## 5. 損失関数とオプティマイザの設定

- **損失関数（Loss Function）**: 交差エントロピー損失（Cross-Entropy Loss）  — 予測確率分布と正解ラベルのずれを測る
- **最適化手法（Optimizer）**: SGD（確率的勾配降下法）+Momentum  — 「ボールが谷を転がる」ように重みを更新する
- **学習率スケジューラ（Learning Rate Scheduler）**: StepLR  — 一定エポックごとに学習率を段階的に下げる


In [ ]:
# ハイパーパラメータ（Hyperparameters）
LEARNING_RATE = 0.1    # 学習率 η（一歩の幅）
MOMENTUM = 0.9         # モメンタム（前回の移動方向を引き継ぐ割合）
WEIGHT_DECAY = 5e-4    # 重み減衰（L2 正則化の係数）
NUM_EPOCHS = 30        # エポック数（全データを何周学習するか）

# 損失関数
criterion = nn.CrossEntropyLoss()

# オプティマイザ: SGD with Momentum
optimizer = optim.SGD(
    model.parameters(),
    lr=LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY
)

# 学習率スケジューラ: 10エポックごとに lr を 0.1 倍に下げる
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

print(f"学習率 (Learning Rate): {LEARNING_RATE}")
print(f"モメンタム (Momentum): {MOMENTUM}")
print(f"重み減衰 (Weight Decay): {WEIGHT_DECAY}")
print(f"エポック数 (Epochs): {NUM_EPOCHS}")


## 6. 学習ループ

1エポックごとに以下を実行します:
1. **訓練**: ミニバッチを順伝播 → 損失計算 → 逆伝播 → 重み更新
2. **評価**: テストデータで精度（Accuracy）を計算
3. **記録**: 損失と精度を記録して学習曲線に使う


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """1エポック分の訓練を行い、平均損失と精度を返す。"""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        # 順伝播（Forward Pass）
        outputs = model(images)
        loss = criterion(outputs, labels)

        # 逆伝播（Backward Pass）+ 重み更新
        optimizer.zero_grad()  # 勾配をリセット
        loss.backward()        # 誤差逆伝播（Backpropagation）
        optimizer.step()       # 重みを更新

        # 統計
        total_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += images.size(0)

    return total_loss / total, 100.0 * correct / total


def evaluate(model, loader, criterion, device):
    """テストデータで損失と精度を評価する。"""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():  # 勾配計算不要（メモリ節約）
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += images.size(0)

    return total_loss / total, 100.0 * correct / total


print("学習ループの関数を定義しました。次のセルで学習を開始します。")


In [ ]:
# 学習の実行
history = {
    "train_loss": [], "train_acc": [],
    "test_loss":  [], "test_acc":  [],
    "lr": []
}

print(f"{'Epoch':>5} {'Train Loss':>10} {'Train Acc':>9} {'Test Loss':>9} {'Test Acc':>8} {'LR':>8}")
print("-" * 55)

for epoch in range(1, NUM_EPOCHS + 1):
    # 訓練
    tr_loss, tr_acc = train_one_epoch(model, trainloader, criterion, optimizer, device)
    # 評価
    te_loss, te_acc = evaluate(model, testloader, criterion, device)
    # 学習率更新
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()

    # 記録
    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["test_loss"].append(te_loss)
    history["test_acc"].append(te_acc)
    history["lr"].append(current_lr)

    if epoch % 5 == 0 or epoch == 1:
        print(f"{epoch:>5} {tr_loss:>10.4f} {tr_acc:>8.2f}% {te_loss:>9.4f} {te_acc:>7.2f}% {current_lr:>8.5f}")

print()
print(f"最終テスト精度: {history['test_acc'][-1]:.2f}%")
print(f"最高テスト精度: {max(history['test_acc']):.2f}% (Epoch {history['test_acc'].index(max(history['test_acc']))+1})")


## 7. 学習曲線の可視化

学習曲線（Learning Curve）を見て、学習が順調かどうかを確認します。

- 訓練損失と検証損失が大きくかい離していたら **過学習（Overfitting）** のサインです
- 両方が高止まりしていたら **未学習（Underfitting）** です


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs = range(1, NUM_EPOCHS + 1)

# 損失曲線
axes[0].plot(epochs, history["train_loss"], label="Train Loss", color="royalblue")
axes[0].plot(epochs, history["test_loss"],  label="Test Loss",  color="tomato")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("損失曲線 (Loss Curve)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 精度曲線
axes[1].plot(epochs, history["train_acc"], label="Train Acc", color="royalblue")
axes[1].plot(epochs, history["test_acc"],  label="Test Acc",  color="tomato")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_title("精度曲線 (Accuracy Curve)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 学習率の推移
axes[2].plot(epochs, history["lr"], color="green")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Learning Rate")
axes[2].set_title("学習率スケジュール (LR Schedule)")
axes[2].grid(True, alpha=0.3)

plt.suptitle("ベースライン CNN の学習結果", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


## 8. モデルの保存

学習済みモデルを Google Drive に保存して次のノートブックで使えるようにします。


In [ ]:
# Google Drive をマウント（必要な場合のみ実行）
# from google.colab import drive
# drive.mount('/content/drive')
# SAVE_DIR = '/content/drive/MyDrive/ml_homework/'

# ローカル保存（Colab セッション内）
import os
SAVE_DIR = './checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

save_path = f'{SAVE_DIR}/baseline_cnn.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'history': history,
    'epoch': NUM_EPOCHS,
    'test_acc': history['test_acc'][-1],
}, save_path)

print(f"モデルを保存しました: {save_path}")
print(f"テスト精度: {history['test_acc'][-1]:.2f}%")


## 9. 試してみよう（課題）

以下の実験を試して、学習への影響を観察してください。

### 課題 1: 学習率を変えてみる
- `LEARNING_RATE = 0.01` に下げると学習が遅くなりますか？
- `LEARNING_RATE = 0.5` に上げると損失が発散しますか？
- 学習率は **一歩の幅** です。大きすぎると谷を飛び越え、小さすぎると進まない。

### 課題 2: ミニバッチサイズを変えてみる
- `BATCH_SIZE = 32` にすると学習が安定しますか？ノイズが多くなりますか？
- `BATCH_SIZE = 512` にすると 1 エポックの時間はどう変わりますか？

### 課題 3: 層の深さを変えてみる
- Conv ブロックを 1 つ追加すると精度は上がりますか？
- Linear 層の中間ユニット数（`512`）を `256` や `1024` に変えてみましょう。
